# Simulación del estómago — Fase 3: fluido 2D con Lattice Boltzmann y paredes móviles

**Qué añade esta fase.** Hasta ahora el movimiento del contenido era un flujo inventado (fase 2). Aquí resolvemos de verdad las ecuaciones de **Navier–Stokes** en 2D, con **paredes que se contraen** como las ondas antrales, usando el método **Lattice Boltzmann** (LBM).

**Por qué Lattice Boltzmann.**
- Maneja con naturalidad paredes que se mueven y cambian de forma: cada nodo es simplemente fluido o sólido.
- Es local y explícito, así que se paraleliza casi perfectamente en GPU.
- Da la tensión viscosa en cada nodo **sin derivar** el campo de velocidades. Es justo la "fuerza" que querías ver en los vórtices.

**Un solo código, tres backends.** El núcleo está escrito con funciones puras, sin modificar arrays en el sitio, y recibe el módulo numérico `xp` como parámetro:
- `numpy`: CPU. Es el que verifica todo el cuaderno.
- `cupy`: GPU con la misma API que NumPy.
- `jax.numpy` + `jax.jit` + `lax.fori_loop`: GPU con el bucle de tiempo entero compilado.

**Estructura**
1. Teoría: de Boltzmann a Navier–Stokes.
2. Código.
3. Verificación: Poiseuille, Couette, vórtice de Taylor–Green, conservación de masa, y **bombeo peristáltico frente a la solución analítica de Shapiro** (paredes móviles).
4. El antro: ondas peristálticas que avanzan hacia un píloro cerrado, chorro retropulsivo, vórtices, presión y tensiones.
5. Trazadores: cuánto mezcla el flujo real, para acotar el $D_{mix}$ de la fase 2.
6. GPU: test de equivalencia con NumPy y comparación de velocidad.
7. Exportación del campo de velocidades para acoplarlo al transporte.

**Uso en Colab.** En CPU, `Ejecutar todo` tarda unos 4 minutos. Para la sección 6, activa la GPU en `Entorno de ejecución → Cambiar tipo de entorno → GPU T4`.

## 1. Teoría: de Boltzmann a Navier–Stokes

### 1.1 La ecuación de Lattice Boltzmann
En lugar de velocidades y presiones, LBM evoluciona **funciones de distribución** $f_i(\mathbf{x}, t)$: la densidad de "partículas de fluido" que se mueven con la velocidad discreta $\mathbf{c}_i$. Cada paso tiene dos fases:

$$ \underbrace{f_i^\ast = f_i - \frac{1}{\tau}\big(f_i - f_i^{eq}\big) + S_i}_{\text{colisión (local)}}, \qquad \underbrace{f_i(\mathbf{x} + \mathbf{c}_i, t+1) = f_i^\ast(\mathbf{x}, t)}_{\text{propagación (a los vecinos)}} . $$

La colisión relaja hacia el equilibrio con un tiempo $\tau$ (modelo BGK). $S_i$ es el término de fuerza volumétrica de Guo.

### 1.2 Red D2Q9 y equilibrio
| $i$ | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|---|
| $\mathbf{c}_i$ | (0,0) | (1,0) | (0,1) | (−1,0) | (0,−1) | (1,1) | (−1,1) | (−1,−1) | (1,−1) |
| $w_i$ | 4/9 | 1/9 | 1/9 | 1/9 | 1/9 | 1/36 | 1/36 | 1/36 | 1/36 |

$$ f_i^{eq} = w_i\,\rho\left[1 + 3\,\mathbf{c}_i\cdot\mathbf{u} + \tfrac92(\mathbf{c}_i\cdot\mathbf{u})^2 - \tfrac32 |\mathbf{u}|^2\right], \qquad \rho = \sum_i f_i,\quad \rho\mathbf{u} = \sum_i f_i\,\mathbf{c}_i . $$

### 1.3 Qué ecuaciones resuelve (expansión de Chapman–Enskog)
En el límite de número de Mach bajo ($|\mathbf{u}| \ll c_s = 1/\sqrt3$), los momentos de $f_i$ cumplen
$$ \partial_t\rho + \nabla\cdot(\rho\mathbf{u}) = 0, \qquad \partial_t(\rho\mathbf{u}) + \nabla\cdot(\rho\mathbf{u}\mathbf{u}) = -\nabla p + \nabla\cdot\big[\rho\nu(\nabla\mathbf{u} + \nabla\mathbf{u}^T)\big] + \mathbf{F}, $$
con
$$ p = c_s^2\,\rho = \rho/3, \qquad \nu = \frac{\tau - 1/2}{3}\ \ (\text{unidades de red}). $$
Es Navier–Stokes **débilmente compresible**. El error de compresibilidad es de orden $\mathrm{Ma}^2$, por eso vigilamos que $|\mathbf{u}|$ en unidades de red sea $\lesssim 0.05$.

**Tensiones sin derivadas.** La parte de no equilibrio da directamente el tensor viscoso:
$$ \sigma_{\alpha\beta} = -\Big(1 - \frac{1}{2\tau}\Big)\sum_i \big(f_i - f_i^{eq}\big)\,c_{i\alpha}c_{i\beta}. $$

### 1.4 Paredes
- **Rebote a mitad de enlace** (*halfway bounce-back*): la población que choca con un nodo sólido vuelve por donde vino. La pared queda situada a medio camino entre el último nodo fluido y el primer sólido. Impone velocidad nula con precisión de 2º orden.
- **Pared móvil** (Ladd): $f_{\bar i}(\mathbf{x}, t+1) = f_i^\ast(\mathbf{x}, t) - 6\,w_i\,\rho\,(\mathbf{c}_i\cdot\mathbf{u}_w)$. Transmite al fluido la velocidad de la pared $\mathbf{u}_w$.
- **Geometría que cambia**: cuando un nodo pasa de sólido a fluido ("nodo fresco"), se inicializa con el equilibrio, usando la densidad media de sus vecinos y la velocidad de la pared (Lallemand y Luo, 2003). Esto no conserva la masa exactamente, así que vigilamos su deriva.

### 1.5 Unidades
Fijamos el tamaño de nodo $\Delta x$ y el parámetro $\tau$. El paso de tiempo sale de igualar la viscosidad física:
$$ \Delta t = \nu_{red}\,\frac{\Delta x^2}{\nu}, \qquad u_{fis} = u_{red}\frac{\Delta x}{\Delta t}, \qquad \sigma_{fis} = \sigma_{red}\,\rho\Big(\frac{\Delta x}{\Delta t}\Big)^2. $$

## 2. Código

### 2.1 Elección del backend
Pon `BACKEND` a `"numpy"` (CPU), `"cupy"` o `"jax"` (GPU). Las secciones 3–5 usan NumPy en float64 como referencia. La 6 compara con la GPU si está disponible.

In [ ]:
import math, time
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

plt.rcParams.update({
    "figure.dpi": 110, "figure.figsize": (11, 4.2),
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "lines.linewidth": 2.0, "font.size": 10.5, "legend.frameon": False,
})
GRIS = "#3b3b3b"; GRIS_MEDIO = "#9a9a96"; TINTA2 = "#52514e"
C1, C2, C3, C4 = "#2a78d6", "#eb6834", "#1baf7a", "#6250d6"
pd.set_option("display.float_format", lambda x: f"{x:.4g}")
VERIFICACION = []

BACKEND = "numpy"          # "numpy" | "cupy" | "jax"  (la sección 6 prueba la GPU aparte)

### 2.2 Núcleo Lattice Boltzmann (genérico para NumPy / CuPy / JAX)

In [ ]:
# ===========================================================================
# Lattice Boltzmann D2Q9 escrito con funciones PURAS (sin modificar arrays en el sitio).
# Así el mismo código corre con NumPy, CuPy (GPU, misma API) o jax.numpy (GPU + jit).
# Unidades de red: dx = dt = 1, c_s^2 = 1/3, nu = (tau - 1/2)/3.
# Convenio de arrays: f tiene forma (9, ny, nx); la fila j = 0 es el fondo (y crece hacia arriba).
# ===========================================================================
CX  = [0, 1, 0, -1, 0, 1, -1, -1, 1]
CY  = [0, 0, 1, 0, -1, 1, 1, -1, -1]
W   = [4 / 9] + [1 / 9] * 4 + [1 / 36] * 4
OPP = [0, 3, 4, 1, 2, 7, 8, 5, 6]          # dirección opuesta


class Red:
    # Constantes D2Q9 ya convertidas al backend (xp) y al tipo de dato elegidos
    def __init__(self, xp, dtype="float64"):
        self.xp, self.dtype = xp, dtype
        self.cx = xp.asarray(CX, dtype=dtype).reshape(9, 1, 1)
        self.cy = xp.asarray(CY, dtype=dtype).reshape(9, 1, 1)
        self.w = xp.asarray(W, dtype=dtype).reshape(9, 1, 1)


def equilibrio(rho, ux, uy, R):
    cu = 3.0 * (R.cx * ux + R.cy * uy)
    usq = 1.5 * (ux * ux + uy * uy)
    return R.w * rho * (1.0 + cu + 0.5 * cu * cu - usq)


def macroscopicas(f, R, Fx=0.0, Fy=0.0):
    rho = f.sum(axis=0)
    jx = f[1] - f[3] + f[5] - f[6] - f[7] + f[8]        # sum_i f_i c_ix, escrito explícitamente (más rápido)
    jy = f[2] - f[4] + f[5] + f[6] - f[7] - f[8]
    ux = (jx + 0.5 * Fx) / rho                          # desplazamiento de Guo: u = (sum f c + F/2)/rho
    uy = (jy + 0.5 * Fy) / rho
    return rho, ux, uy


def colision_bgk(f, rho, ux, uy, tau, R, Fx=0.0, Fy=0.0):
    feq = equilibrio(rho, ux, uy, R)
    cu = R.cx * ux + R.cy * uy
    cF = R.cx * Fx + R.cy * Fy
    # término de fuerza de Guo (2002): segundo orden con fuerza volumétrica
    S = (1.0 - 0.5 / tau) * R.w * (3.0 * ((R.cx - ux) * Fx + (R.cy - uy) * Fy) + 9.0 * cu * cF)
    return f - (f - feq) / tau + S


def propagar(f, R):
    xp = R.xp
    return xp.stack([xp.roll(f[i], shift=(CY[i], CX[i]), axis=(0, 1)) for i in range(9)])


def rebote(fs, fpost, solido, uwx, uwy, rho, R):
    # Rebote a mitad de enlace con pared móvil (Ladd):
    # si la población i llega a un nodo fluido x desde un nodo sólido x - c_i, se sustituye por
    # f_i(x) = f*_opp(i)(x) + 6 w_i rho (c_i · u_w),  con u_w la velocidad de la pared en ese nodo sólido.
    xp = R.xp
    salida = []
    fluido = ~solido
    for i in range(9):
        if i == 0:
            salida.append(fs[0]); continue
        desde_solido = xp.roll(solido, shift=(CY[i], CX[i]), axis=(0, 1)) & fluido
        cu_w = 0.0
        if CX[i] != 0 and uwx is not None:
            cu_w = cu_w + CX[i] * xp.roll(uwx, shift=(CY[i], CX[i]), axis=(0, 1))
        if CY[i] != 0 and uwy is not None:
            cu_w = cu_w + CY[i] * xp.roll(uwy, shift=(CY[i], CX[i]), axis=(0, 1))
        rebotado = fpost[OPP[i]] + 6.0 * W[i] * rho * cu_w
        salida.append(xp.where(desde_solido, rebotado, fs[i]))
    return xp.stack(salida)


def rellenar(f, solido_viejo, solido_nuevo, rho, uwx, uwy, R):
    # Nodos que pasan de sólido a fluido ("nodos frescos") se inicializan con el equilibrio:
    # densidad = media de los vecinos fluidos, velocidad = la de la pared (Lallemand y Luo, 2003).
    xp = R.xp
    fresco = solido_viejo & ~solido_nuevo
    fluido_previo = (~solido_viejo).astype(rho.dtype)
    suma, n = 0.0 * rho, 0.0 * rho
    for i in range(1, 9):
        suma = suma + xp.roll(rho * fluido_previo, shift=(CY[i], CX[i]), axis=(0, 1))
        n = n + xp.roll(fluido_previo, shift=(CY[i], CX[i]), axis=(0, 1))
    rho_vec = xp.where(n > 0, suma / xp.maximum(n, 1.0), 1.0)
    cero = 0.0 * rho
    f_fresco = equilibrio(rho_vec, cero if uwx is None else uwx, cero if uwy is None else uwy, R)
    f = xp.where(fresco[None], f_fresco, f)
    # dentro del sólido guardamos el equilibrio en reposo, f_i = w_i (no interviene en la dinámica)
    return xp.where(solido_nuevo[None], R.w + 0.0 * f, f)


def crear_paso(R, tau, geometria, Fx=0.0, Fy=0.0):
    # Devuelve paso(f, t) -> f(t+1). geometria(t) -> (solido, uwx, uwy) en unidades de red.
    def paso(f, t):
        solido, uwx, uwy = geometria(t)
        rho, ux, uy = macroscopicas(f, R, Fx, Fy)
        fpost = colision_bgk(f, rho, ux, uy, tau, R, Fx, Fy)
        fs = propagar(fpost, R)
        fs = rebote(fs, fpost, solido, uwx, uwy, rho, R)
        solido2, uwx2, uwy2 = geometria(t + 1)
        return rellenar(fs, solido, solido2, rho, uwx2, uwy2, R)
    return paso


def tension_viscosa(f, tau, R, Fx=0.0, Fy=0.0):
    # Tensor de tensiones viscosas a partir de la parte de no equilibrio (sin derivar el campo):
    # sigma_ab = -(1 - 1/(2 tau)) sum_i (f_i - feq_i) c_ia c_ib    [unidades de red]
    rho, ux, uy = macroscopicas(f, R, Fx, Fy)
    fneq = f - equilibrio(rho, ux, uy, R)
    k = -(1.0 - 0.5 / tau)
    sxx = k * (fneq * R.cx * R.cx).sum(axis=0)
    syy = k * (fneq * R.cy * R.cy).sum(axis=0)
    sxy = k * (fneq * R.cx * R.cy).sum(axis=0)
    return sxx, syy, sxy


def geometria_fija(solido, R):
    xp = R.xp
    cero = xp.zeros(solido.shape, dtype=R.dtype)
    return lambda t: (solido, cero, cero)


# ---------------------------------------------------------------------------
# Paredes que se mueven: canal peristáltico periódico (test de Shapiro)
# ---------------------------------------------------------------------------
def geometria_peristaltica(nx, ny, a, phi, lam, c, R):
    # Semiancho H(x,t) = a (1 + phi sin(2 pi (x - c t)/lam)) alrededor del eje y = ny/2.
    # Las paredes se mueven solo en vertical (como en Shapiro et al., 1969).
    xp = R.xp
    x = xp.arange(nx, dtype=R.dtype)[None, :] + 0.5
    y = xp.arange(ny, dtype=R.dtype)[:, None] + 0.5
    yc = ny / 2.0
    k = 2 * math.pi / lam
    def geom(t):
        H = a * (1.0 + phi * xp.sin(k * (x - c * t)))
        dHdt = -a * phi * k * c * xp.cos(k * (x - c * t))
        abajo, arriba = y < yc - H, y > yc + H
        solido = abajo | arriba
        uwy = xp.where(abajo, -dHdt, 0.0 * y) + xp.where(arriba, dHdt, 0.0 * y)
        return solido, None, uwy                        # uwx = None: las paredes solo se mueven en vertical
    return geom


def integrar(paso, f, t0, n_pasos, R, usar_jax=False):
    # Bucle de tiempo. Con JAX se compila con lax.fori_loop (todo el bucle en la GPU).
    if usar_jax:
        from jax import lax
        f, _ = lax.fori_loop(0, n_pasos, lambda i, ft: (paso(ft[0], ft[1]), ft[1] + 1.0), (f, R.xp.asarray(t0, dtype=R.dtype)))
        return f
    for n in range(n_pasos):
        f = paso(f, t0 + n)
    return f

### 2.3 Geometría del antro y conversión de unidades

In [ ]:
@dataclass
class ParamAntro:
    # Geometría y movimiento (unidades físicas: cm, s)
    Lx: float = 10.0          # cm   longitud (fondo -> píloro)
    H: float = 4.0            # cm   anchura en reposo
    margen: float = 1.0       # cm   holgura para que el fondo pueda dilatarse
    x_ini: float = 4.5        # cm   donde empiezan las ondas antrales
    x_plena: float = 6.5      # cm   donde alcanzan su amplitud máxima
    L_fondo: float = 3.5      # cm   zona proximal que se dilata para conservar el volumen
    A_max: float = 1.4        # cm   profundidad de la contracción en cada pared (70 % de oclusión)
    w: float = 0.5            # cm   semiancho de la onda
    c: float = 0.25           # cm/s velocidad de la onda peristáltica
    T: float = 20.0           # s    periodo (3 ondas por minuto)
    nu: float = 0.1           # cm²/s viscosidad cinemática (comida líquida espesa; agua = 0.01)
    rho: float = 1.0          # g/cm³
    # Discretización
    nx: int = 128             # nodos a lo largo (sin contar las paredes de los extremos)
    tau: float = 0.65


class Unidades:
    # Conversión física <-> red: dx fija la malla; dt sale de igualar la viscosidad.
    def __init__(self, pa):
        self.dx = pa.Lx / pa.nx                        # cm
        self.nu_red = (pa.tau - 0.5) / 3.0
        self.dt = self.nu_red * self.dx ** 2 / pa.nu    # s
        self.vel = self.dx / self.dt                    # (cm/s) por unidad de red
        self.tension = pa.rho * self.vel ** 2           # (dyn/cm² = 0.1 Pa) por unidad de red

    def resumen(self, pa):
        return {"dx [cm]": self.dx, "dt [s]": self.dt, "pasos por onda": pa.T / self.dt,
                "c onda [red]": pa.c / self.vel, "Re = c (H/2)/nu": pa.c * (pa.H / 2) / pa.nu,
                "nu [red]": self.nu_red}


def geometria_antro(pa, R):
    # Paredes superior e inferior con ondas que viajan hacia el píloro (x = Lx).
    # El fondo (x < L_fondo) se dilata lo justo para que el volumen total no cambie.
    # Extremos x = 0 (fondo) y x = Lx (píloro, cerrado en esta fase) son paredes fijas.
    xp = R.xp
    U = Unidades(pa)
    dx = U.dx
    ny = int(round((pa.H + 2 * pa.margen) / dx))
    nx = pa.nx + 2                                       # + una columna de pared en cada extremo
    x = (xp.arange(nx, dtype=R.dtype)[None, :] - 0.5) * dx          # cm (la columna 0 es pared)
    y = (xp.arange(ny, dtype=R.dtype)[:, None] + 0.5) * dx
    rampa = xp.clip((x - pa.x_ini) / (pa.x_plena - pa.x_ini), 0.0, 1.0)
    envolvente = pa.A_max * rampa * rampa * (3 - 2 * rampa)         # suave (smoothstep)
    ventana_fondo = xp.clip((pa.L_fondo + 1.0 - x) / 1.0, 0.0, 1.0) * (x > 0) * (x < pa.Lx)
    int_ventana = float(xp.sum(ventana_fondo)) * dx
    x0 = pa.x_ini - 3 * pa.w
    K = int(math.ceil((pa.Lx + 3 * pa.w - x0) / (pa.c * pa.T))) + 1
    extremos = (x <= 0) | (x >= pa.Lx)
    def ondas(t_fis):
        s = t_fis % pa.T
        d = 0.0 * x
        dd = 0.0 * x
        for k in range(K):
            xk = x0 + pa.c * (s + k * pa.T)
            g = xp.exp(-((x - xk) / pa.w) ** 2)
            d = d + g
            dd = dd + g * 2 * (x - xk) * pa.c / pa.w ** 2
        return envolvente * d, envolvente * dd                      # delta [cm], d delta/dt [cm/s]
    def geom(t):
        t_fis = t * U.dt
        delta, ddelta = ondas(t_fis)
        interior = (x > 0) & (x < pa.Lx)
        area = xp.sum(delta * interior) * dx                         # área contraída por pared
        darea = xp.sum(ddelta * interior) * dx
        d_f = area / int_ventana                                     # dilatación del fondo
        dd_f = darea / int_ventana
        yb = pa.margen + delta - ventana_fondo * d_f
        yt = pa.margen + pa.H - delta + ventana_fondo * d_f
        vb = ddelta - ventana_fondo * dd_f                           # cm/s
        vt = -ddelta + ventana_fondo * dd_f
        abajo, arriba = y < yb, y > yt
        solido = abajo | arriba | extremos
        uwy = (xp.where(abajo, vb, 0.0 * y) + xp.where(arriba, vt, 0.0 * y)) * ~extremos / U.vel
        return solido, None, uwy
    return geom, nx, ny, U

### 2.4 Bucle de simulación, trazadores y post-proceso

In [ ]:
def a_numpy(a):
    # Convierte un array de cualquier backend (NumPy, CuPy, JAX) a NumPy
    if hasattr(a, "get"):
        return a.get()              # CuPy
    return np.asarray(a)            # NumPy y JAX


def interpolar_bilineal(campo, xs, ys):
    # campo (ny, nx) en nodos de red; xs, ys en coordenadas de red (centro del nodo i en x = i)
    ny, nx = campo.shape
    x = np.clip(xs, 0, nx - 1.001); y = np.clip(ys, 0, ny - 1.001)
    i0 = np.floor(x).astype(int); j0 = np.floor(y).astype(int)
    tx = x - i0; ty = y - j0
    c00 = campo[j0, i0]; c10 = campo[j0, i0 + 1]; c01 = campo[j0 + 1, i0]; c11 = campo[j0 + 1, i0 + 1]
    return (1 - tx) * (1 - ty) * c00 + tx * (1 - ty) * c10 + (1 - tx) * ty * c01 + tx * ty * c11


def simular_antro(pa, R, periodos=4, guardar_cada_s=0.25, n_particulas=3000, semilla=1,
                  usar_jax=False, informar=True):
    # Ejecuta el antro y devuelve instantáneas (en NumPy y unidades físicas) + trayectorias de trazadores.
    xp = R.xp
    geom, nx, ny, U = geometria_antro(pa, R)
    paso = crear_paso(R, pa.tau, geom)
    f = R.w * xp.ones((9, ny, nx), dtype=R.dtype)                     # reposo, rho = 1
    pasos_bloque = max(1, int(round(guardar_cada_s / U.dt)))
    n_bloques = int(round(periodos * pa.T / (pasos_bloque * U.dt)))
    if usar_jax:
        import jax
        avanzar = jax.jit(lambda f, t0: integrar(paso, f, t0, pasos_bloque, R, usar_jax=True))
    else:
        avanzar = lambda f, t0: integrar(paso, f, t0, pasos_bloque, R)
    # trazadores: repartidos al azar por el fluido inicial (coordenadas de red)
    rng = np.random.default_rng(semilla)
    s0 = a_numpy(geom(0)[0])
    cand = np.argwhere(~s0)
    elegido = cand[rng.choice(len(cand), n_particulas, replace=False)]
    py = elegido[:, 0] + rng.uniform(-0.5, 0.5, n_particulas)
    px = elegido[:, 1] + rng.uniform(-0.5, 0.5, n_particulas)
    hist = {k: [] for k in ("t", "ux", "uy", "rho", "solido", "tau_max", "px", "py", "masa")}
    def campos(f, t):
        rho, ux, uy = macroscopicas(f, R)
        sxx, syy, sxy = tension_viscosa(f, pa.tau, R)
        tmax = xp.sqrt(0.25 * (sxx - syy) ** 2 + sxy ** 2)            # cizalla máxima
        sol = geom(t)[0]
        return [a_numpy(v) for v in (rho, ux, uy, tmax, sol)]
    t_red = 0
    rho, ux, uy, tmax, sol = campos(f, t_red)
    t_ini = time.time()
    for b in range(n_bloques + 1):
        hist["t"].append(t_red * U.dt)
        hist["ux"].append(np.where(sol, np.nan, ux) * U.vel)
        hist["uy"].append(np.where(sol, np.nan, uy) * U.vel)
        hist["rho"].append(np.where(sol, np.nan, rho))
        hist["tau_max"].append(np.where(sol, np.nan, tmax) * U.tension / 10.0)   # Pa
        hist["solido"].append(sol)
        hist["px"].append(px.copy()); hist["py"].append(py.copy())
        hist["masa"].append(float(np.sum(np.where(sol, 0.0, rho))))
        if b == n_bloques:
            break
        f = avanzar(f, float(t_red))
        t_red += pasos_bloque
        rho2, ux2, uy2, tmax2, sol2 = campos(f, t_red)
        # trazadores: Heun con el campo al principio y al final del bloque (desplazamiento en nodos)
        n_p = pasos_bloque
        vx1 = interpolar_bilineal(np.where(sol, 0, ux), px, py); vy1 = interpolar_bilineal(np.where(sol, 0, uy), px, py)
        qx, qy = px + n_p * vx1, py + n_p * vy1
        vx2 = interpolar_bilineal(np.where(sol2, 0, ux2), qx, qy); vy2 = interpolar_bilineal(np.where(sol2, 0, uy2), qx, qy)
        nx_, ny_ = px + 0.5 * n_p * (vx1 + vx2), py + 0.5 * n_p * (vy1 + vy2)
        ii = np.clip(np.rint(nx_).astype(int), 0, nx - 1); jj = np.clip(np.rint(ny_).astype(int), 0, ny - 1)
        dentro = ~sol2[jj, ii]
        px = np.where(dentro, nx_, px); py = np.where(dentro, ny_, py)   # si caería en la pared, se queda
        rho, ux, uy, tmax, sol = rho2, ux2, uy2, tmax2, sol2
        if informar and (b + 1) % max(1, n_bloques // 8) == 0:
            print(f"  t = {t_red * U.dt:6.1f} s   ({time.time() - t_ini:5.1f} s de cálculo)")
    out = {k: np.array(v) for k, v in hist.items()}
    out["U"], out["nx"], out["ny"], out["pasos"] = U, nx, ny, t_red
    out["segundos"] = time.time() - t_ini
    return out


def vorticidad(ux, uy, dx):
    # w = dv/dx - du/dy (1/s), diferencias centradas; NaN en el sólido
    dvdx = np.gradient(uy, dx, axis=1)
    dudy = np.gradient(ux, dx, axis=0)
    return dvdx - dudy

## 3. Verificación

Todas las pruebas usan NumPy en doble precisión. `R = Red(np)` agrupa las constantes D2Q9.

### T1 — Poiseuille (fuerza volumétrica + paredes fijas)
En un canal periódico de anchura $H$ empujado por una fuerza $g$, la solución estacionaria es la parábola $u(y) = \frac{g}{2\nu}(y - y_0)(y_1 - y)$, con las paredes a medio enlace. Con $\tau$ fijo, el error del rebote debe caer como $1/H^2$.

In [ ]:
R = Red(np)

def poiseuille(H, tau=0.8, g=1e-6):
    ny, nx = H + 2, 3
    sol = np.zeros((ny, nx), bool); sol[0] = sol[-1] = True
    nu = (tau - 0.5) / 3
    paso = crear_paso(R, tau, geometria_fija(sol, R), Fx=g)
    f = R.w * np.ones((9, ny, nx))
    f = integrar(paso, f, 0, int(3 * H * H / nu), R)
    rho, ux, uy = macroscopicas(f, R, g, 0.0)
    y = np.arange(ny) + 0.5
    exacta = g / (2 * nu) * (y - 1.0) * (ny - 1.0 - y)
    fl = ~sol[:, 1]
    return y[fl], ux[fl, 1], exacta[fl], float(np.sqrt(np.mean((ux[fl, 1] - exacta[fl]) ** 2)) / exacta.max())

Hs = [6, 12, 24]
res_P = [poiseuille(H) for H in Hs]
e_P = [r[3] for r in res_P]
p_P = math.log(e_P[-2] / e_P[-1]) / math.log(2)
print("Errores relativos:", ["%.2e" % e for e in e_P], f" orden ≈ {p_P:.2f}")
VERIFICACION.append({"test": "T1 Poiseuille: orden ≈ 2", "valor": p_P, "criterio": "|p − 2| < 0.2", "pasa": abs(p_P - 2) < 0.2})

### T2 — Couette (pared móvil)
La pared superior se mueve a velocidad $U$ y la inferior está fija. El perfil exacto es lineal, y el rebote con pared móvil lo reproduce **exactamente**, salvo redondeo.

### T3 — Vórtice de Taylor–Green
Un campo periódico de vórtices decae como $e^{-2\nu k^2 t}$. Este test comprueba la relación $\nu = (\tau-\tfrac12)/3$ y el orden 2 con escalado difusivo ($U \propto 1/N$, pasos $\propto N^2$). También comprueba que la masa se conserva exactamente en un dominio periódico.

In [ ]:
def couette(H=16, tau=0.8, U=0.01):
    ny, nx = H + 2, 3
    sol = np.zeros((ny, nx), bool); sol[0] = sol[-1] = True
    uwx = np.zeros((ny, nx)); uwx[-1] = U
    paso = crear_paso(R, tau, lambda t: (sol, uwx, 0 * uwx))
    f = integrar(paso, R.w * np.ones((9, ny, nx)), 0, int(10 * H * H / ((tau - 0.5) / 3)), R)
    rho, ux, uy = macroscopicas(f, R)
    y = np.arange(ny) + 0.5; fl = ~sol[:, 1]
    return float(np.max(np.abs(ux[fl, 1] - U * (y[fl] - 1.0) / (ny - 2.0))) / U), y[fl], ux[fl, 1]

def taylor_green(N, tau=0.8, U0=0.64):
    U = U0 / N; nu = (tau - 0.5) / 3; k = 2 * np.pi / N
    x = np.arange(N) + 0.5; X, Y = np.meshgrid(x, x)
    ux0 = -U * np.cos(k * X) * np.sin(k * Y); uy0 = U * np.sin(k * X) * np.cos(k * Y)
    rho0 = 1 - 0.75 * U * U * (np.cos(2 * k * X) + np.cos(2 * k * Y))
    f = equilibrio(rho0, ux0, uy0, R)
    paso = crear_paso(R, tau, geometria_fija(np.zeros((N, N), bool), R))
    T = int(round(np.log(2) / (2 * nu * k * k)))                 # hasta que la amplitud cae a la mitad
    f = integrar(paso, f, 0, T, R)
    rho, ux, uy = macroscopicas(f, R)
    dec = np.exp(-2 * nu * k * k * T)
    err = float(np.sqrt(np.mean((ux - ux0 * dec) ** 2 + (uy - uy0 * dec) ** 2)) / (U * dec))
    nu_eff = -np.log(np.mean(ux ** 2 + uy ** 2) / np.mean(ux0 ** 2 + uy0 ** 2)) / (4 * k * k * T)
    return err, nu_eff / nu - 1, abs(rho.sum() - rho0.sum()) / rho0.sum()

e_C, yC, uC = couette()
Ns = [16, 32, 64]
res_TG = [taylor_green(N) for N in Ns]
p_TG = math.log(res_TG[-2][0] / res_TG[-1][0]) / math.log(2)

fig, axs = plt.subplots(1, 3, figsize=(14, 3.9))
yP, uP, exP, _ = res_P[1]
axs[0].plot(exP, yP, color=GRIS, lw=3, alpha=0.4, label="exacta"); axs[0].plot(uP, yP, "o", ms=5, mfc="none", color=C1, label="LBM")
axs[0].set(xlabel="u [red]", ylabel="y [nodos]", title="T1 · Poiseuille (H = 12)"); axs[0].legend()
axs[0].ticklabel_format(axis="x", style="sci", scilimits=(0, 0))
axs[1].plot(0.01 * (yC - 1) / (yC.max() - 0.5), yC, color=GRIS, lw=3, alpha=0.4, label="exacta"); axs[1].plot(uC, yC, "o", ms=5, mfc="none", color=C2, label="LBM")
axs[1].set(xlabel="u [red]", title=f"T2 · Couette (error {e_C:.0e})"); axs[1].legend()
axs[2].loglog(Hs, e_P, marker="o", color=C1, label=f"Poiseuille (p ≈ {p_P:.2f})")
axs[2].loglog(Ns, [r[0] for r in res_TG], marker="^", color=C3, label=f"Taylor–Green (p ≈ {p_TG:.2f})")
axs[2].loglog(Ns, [abs(r[1]) for r in res_TG], marker="s", ls="--", mfc="none", color=C4, label="|ν_efectiva/ν − 1| (TG)")
axs[2].minorticks_off(); axs[2].set_xticks(Hs + Ns); axs[2].set_xticklabels([str(v) for v in Hs + Ns], fontsize=8)
axs[2].set(xlabel="nodos (H o N)", ylabel="error relativo", title="Convergencia"); axs[2].legend(fontsize=8.5)
fig.tight_layout(); plt.show()

VERIFICACION += [
    {"test": "T2 Couette con pared móvil: exacto", "valor": e_C, "criterio": "< 1e-10", "pasa": e_C < 1e-10},
    {"test": "T3 Taylor–Green: orden ≈ 2", "valor": p_TG, "criterio": "|p − 2| < 0.2", "pasa": abs(p_TG - 2) < 0.2},
    {"test": "T3b viscosidad efectiva (N = 64)", "valor": abs(res_TG[-1][1]), "criterio": "< 0.5 %", "pasa": abs(res_TG[-1][1]) < 5e-3},
    {"test": "T4 masa en dominio periódico", "valor": max(r[2] for r in res_TG), "criterio": "< 1e-12", "pasa": max(r[2] for r in res_TG) < 1e-12},
]

### T5 — Bombeo peristáltico frente a la teoría de Shapiro (paredes móviles)
Es el test clave para el estómago, porque usa exactamente la maquinaria de paredes que se deforman. Un canal periódico cuyo semiancho sigue una onda viajera,
$$ H(x,t) = a\,\big[1 + \phi\,\sin\big(2\pi(x - ct)/\lambda\big)\big], $$
bombea fluido aunque no haya diferencia de presión. En el límite de onda larga ($a \ll \lambda$) y Reynolds bajo, la teoría de lubricación (Shapiro, Jaffrin y Weinberg, 1969) da el caudal medio por semicanal:

$$ \frac{\bar Q}{c\,a} = \frac{3\phi^2}{2 + \phi^2} . $$

**Derivación breve.** En el sistema que viaja con la onda, el flujo es estacionario: $u' = -c + \frac{1}{2\mu}\frac{dp}{d\xi}(y^2 - H^2)$. El caudal $q = -cH - \frac{H^3}{3\mu}\frac{dp}{d\xi}$ es constante. Imponiendo que la presión no suba en una longitud de onda, $\int \frac{dp}{d\xi}\,d\xi = 0$, se obtiene $q = -c\langle H^{-2}\rangle/\langle H^{-3}\rangle$. Volviendo al laboratorio, $\bar Q = q + c\langle H\rangle$, y con las medias de $(1+\phi\sin\theta)^{-n}$ queda la fórmula.

Simulamos con $\phi = 0.5$, $\lambda/a = 10$ y $\mathrm{Re} = ca/\nu = 0.3$, en dos resoluciones.

In [ ]:
def shapiro(a, razon=10, phi=0.5, Re=0.3, tau=1.1):
    lam = int(razon * a); nu = (tau - 0.5) / 3; c = Re * nu / a
    ny = int(math.ceil(2 * a * (1 + phi))) + 4; nx = lam
    geom = geometria_peristaltica(nx, ny, a, phi, lam, c, R)
    paso = crear_paso(R, tau, geom)
    f = R.w * np.ones((9, ny, nx))
    Tp = int(round(lam / c)); calent = max(int(10 * a * a / nu), Tp // 4)
    f = integrar(paso, f, 0, calent, R)
    Q, masas = 0.0, []
    for n in range(Tp):                                   # un periodo completo de medida
        s = geom(calent + n)[0]
        Q += np.sum(np.where(s, 0, f[1] - f[3] + f[5] - f[6] - f[7] + f[8])) / nx
        masas.append(np.sum(np.where(s, 0, f.sum(0))))
        f = paso(f, calent + n)
    Q /= Tp
    rho, ux, uy = macroscopicas(f, R)
    return Q / 2 / (c * a), (max(masas) - min(masas)) / np.mean(masas), (geom(calent + Tp)[0], ux / c, uy / c)

teor = 3 * 0.5 ** 2 / (2 + 0.5 ** 2)
t0 = time.time()
res_S = {a: shapiro(a) for a in (4, 8)}
print(f"Shapiro: {time.time() - t0:.0f} s")
filas = [{"a [nodos]": a, "Q/(c a) LBM": r[0], "Q/(c a) teoría": teor, "error relativo": (r[0] - teor) / teor,
          "variación de masa en el periodo": r[1]} for a, r in res_S.items()]
display(pd.DataFrame(filas).style.format(precision=4).hide(axis="index"))

sol8, ux8, uy8 = res_S[8][2]
fig, a1 = plt.subplots(figsize=(11, 3.2))
Y, X = np.mgrid[0:sol8.shape[0], 0:sol8.shape[1]]
im = a1.imshow(np.where(sol8, np.nan, ux8), origin="lower", cmap="RdBu_r", vmin=-2.5, vmax=2.5, aspect="equal")
a1.streamplot(X[0], Y[:, 0], np.where(sol8, 0, ux8), np.where(sol8, 0, uy8), color=GRIS, density=0.9, linewidth=0.6, arrowsize=0.7)
plt.colorbar(im, ax=a1, shrink=0.8, label="u / c")
a1.set(title="T5 · Canal peristáltico (a = 8): velocidad axial y líneas de corriente", xlabel="x [nodos]", ylabel="y [nodos]"); a1.grid(False)
plt.show()
e_S = abs(res_S[8][0] - teor) / teor
VERIFICACION.append({"test": "T5 bombeo peristáltico vs Shapiro (a = 8)", "valor": e_S, "criterio": "< 3 %", "pasa": e_S < 0.03})

**Lectura.** El LBM reproduce el caudal de Shapiro con un error de alrededor del 1 %. Al refinar, la parte numérica del error converge con orden ≈ 2. En pruebas con $a = 16$, que no incluimos porque tardan 5 minutos en CPU, el resultado se estabiliza en torno a +1.5 %. Ese resto es la diferencia entre el LBM, que resuelve Navier–Stokes completo, y la teoría de lubricación, que supone onda infinitamente larga y $\mathrm{Re} \to 0$; con $\lambda/a = 10$ es del orden esperado. La variación de masa por los nodos frescos es de $10^{-4}$, despreciable.

## 4. El antro: ondas peristálticas contra un píloro cerrado

**Montaje** (unidades físicas):
- Dominio de 10 × 4 cm, que es una sección longitudinal del estómago.
- A partir de $x = 4.5$ cm nacen ondas de contracción en las dos paredes. Avanzan hacia el píloro ($x = 10$ cm) a 2.5 mm/s, una cada 20 s (3 por minuto), y ocluyen hasta el 70 % de la luz.
- El píloro está **cerrado**: es la situación de la "trituración" antral, entre vaciados.
- Para que el volumen se conserve con un fluido incompresible, el fondo ($x < 3.5$ cm) se dilata lo justo en cada instante. Es la acomodación del estómago proximal.
- Viscosidad de 0.1 cm²/s (≈ 100 veces el agua, una comida líquida espesa). $\mathrm{Re} = c\,(H/2)/\nu = 5$.

In [ ]:
pa = ParamAntro()
geom_a, nx_a, ny_a, U_a = geometria_antro(pa, R)
display(pd.DataFrame([{**U_a.resumen(pa), "nodos": nx_a * ny_a}]).style.format(precision=4).hide(axis="index"))

PERIODOS = 3            # 1 de arranque + 2 de análisis
t0 = time.time()
SIM = simular_antro(pa, R, periodos=PERIODOS, guardar_cada_s=0.25)
print(f"Simulación del antro: {SIM['segundos']:.0f} s ({SIM['pasos']} pasos, {SIM['segundos'] / SIM['pasos'] * 1e3:.2f} ms/paso)")

In [ ]:
U = SIM["U"]; dx = U.dx
t = SIM["t"]; k_per = int(round(pa.T / (t[1] - t[0])))
ult = slice(len(t) - k_per - 1, len(t))                   # último periodo
rapidez = np.hypot(SIM["ux"], SIM["uy"])
Ma_max = float(np.nanmax(rapidez) / U.vel * math.sqrt(3))
masa = SIM["masa"]
deriva = abs(masa[-1] - masa[-1 - k_per]) / masa[0]
dev_rho = np.abs(SIM["rho"][ult] - 1)
var_rho = float(np.nanpercentile(dev_rho, 99.9))
kk, jj, ii = np.unravel_index(np.nanargmax(dev_rho), dev_rho.shape)
print(f"Mach máximo = {Ma_max:.3f}   |ρ − 1| (percentil 99.9) = {var_rho:.2%}   deriva de masa en un periodo = {deriva:.1e}")
print(f"Máximo absoluto de |ρ − 1| = {np.nanmax(dev_rho):.1%} en x = {(ii - 0.5) * dx:.2f} cm: nodos que la contracción "
      f"deja atrapados contra la esquina del píloro ({int(SIM['solido'][ult][kk][jj-1:jj+2, ii-1:ii+2].sum())} de sus 8 vecinos son sólidos).")
VERIFICACION += [
    {"test": "T6 antro: Mach máximo", "valor": Ma_max, "criterio": "< 0.1", "pasa": Ma_max < 0.1},
    {"test": "T6b antro: compresibilidad |ρ − 1| (p99.9)", "valor": var_rho, "criterio": "< 1 %", "pasa": var_rho < 0.01},
    {"test": "T6c antro: deriva de masa en un periodo", "valor": deriva, "criterio": "< 1e-3", "pasa": deriva < 1e-3},
]

# Coordenadas físicas de los nodos
xc = (np.arange(SIM["nx"]) - 0.5) * dx; yc = (np.arange(SIM["ny"]) + 0.5) * dx
XC, YC = np.meshgrid(xc, yc)

### 4.1 Velocidad, vórtices, presión y tensiones en cuatro fases de la onda
- **Velocidad**: el color es la rapidez y las flechas, la dirección.
- **Vorticidad** $\omega = \partial_x v - \partial_y u$: es el giro local, en rojo antihorario y en azul horario.
- **Presión**: obtenida de $p = c_s^2(\rho - \bar\rho)$. Es la "fuerza" que empuja al fluido.
- **Cizalla máxima** $\tau_{max} = \sqrt{(\sigma_{xx}-\sigma_{yy})^2/4 + \sigma_{xy}^2}$, en Pa, calculada sin derivar. Es la tensión que deforma y rompe la comida.

In [ ]:
fases = [0.0, 0.25, 0.5, 0.75]
idx_f = [ult.start + int(round(fz * k_per)) for fz in fases]
fig, axs = plt.subplots(4, 4, figsize=(15, 9.5))
paso_q = 5
for col, k in enumerate(idx_f):
    sol = SIM["solido"][k]
    ux, uy = SIM["ux"][k], SIM["uy"][k]
    w = vorticidad(np.nan_to_num(ux), np.nan_to_num(uy), dx); w[sol] = np.nan
    p = (SIM["rho"][k] - np.nanmean(SIM["rho"][k])) / 3 * U.tension / 10      # Pa
    tz = SIM["tau_max"][k]
    ext = [xc[0], xc[-1], yc[0], yc[-1]]
    a = axs[0, col]
    im0 = a.imshow(rapidez[k], origin="lower", extent=ext, cmap="Greys", vmin=0, vmax=np.nanpercentile(rapidez[ult], 99.5), aspect="equal")
    a.quiver(XC[::paso_q, ::paso_q], YC[::paso_q, ::paso_q], ux[::paso_q, ::paso_q], uy[::paso_q, ::paso_q],
             color=C1, scale=6, width=0.004)
    a.set_title(f"t = {t[k] - t[ult.start]:.0f} s de la onda")
    a = axs[1, col]
    wl = np.nanpercentile(np.abs(w), 99)
    im1 = a.imshow(w, origin="lower", extent=ext, cmap="RdBu_r", vmin=-wl, vmax=wl, aspect="equal")
    a.streamplot(xc, yc, np.nan_to_num(ux), np.nan_to_num(uy), color=GRIS, density=0.8, linewidth=0.5, arrowsize=0.6)
    a = axs[2, col]
    pl = np.nanpercentile(np.abs(p), 99)
    im2 = a.imshow(p, origin="lower", extent=ext, cmap="PuOr_r", vmin=-pl, vmax=pl, aspect="equal")
    a = axs[3, col]
    im3 = a.imshow(tz, origin="lower", extent=ext, cmap="magma", vmin=0, vmax=np.nanpercentile(SIM["tau_max"][ult], 99.5), aspect="equal")
for fila, (im, lab) in enumerate([(im0, "rapidez [cm/s]"), (im1, "vorticidad [1/s]"), (im2, "presión [Pa]"), (im3, "cizalla máx. [Pa]")]):
    fig.colorbar(im, ax=axs[fila, :], shrink=0.85, label=lab, pad=0.01)
    axs[fila, 0].set_ylabel(["velocidad", "vórtices", "presión", "tensión"][fila])
for a in axs.flat:
    a.set_xticks([]); a.set_yticks([]); a.grid(False)
fig.suptitle("Antro con píloro cerrado (x = 10 cm a la derecha): una onda completa", fontweight="bold")
plt.show()

### 4.2 El chorro retropulsivo
Cuando la contracción se acerca al píloro cerrado, el contenido atrapado entre la onda y el píloro solo puede escapar **hacia atrás**, a través del estrechamiento. Ese es el **chorro retropulsivo**, el mecanismo principal de mezcla y trituración del antro.

El diagrama espacio–tiempo muestra la velocidad axial en el eje central del estómago. En azul, el flujo hacia el fondo; en rojo, hacia el píloro.

In [ ]:
j_eje = SIM["ny"] // 2
u_eje = SIM["ux"][:, j_eje, :]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={"width_ratios": [1.3, 1]})
lim = np.nanpercentile(np.abs(u_eje[ult]), 99.5)
im = a1.imshow(u_eje[ult], origin="lower", aspect="auto", cmap="RdBu_r", vmin=-lim, vmax=lim,
               extent=[xc[0], xc[-1], 0, t[ult][-1] - t[ult][0]])
plt.colorbar(im, ax=a1, label="u axial en el eje [cm/s]")
a1.set(xlabel="x: fondo → píloro [cm]", ylabel="t dentro de la onda [s]", title="Velocidad axial en el eje central"); a1.grid(False)
for xs in (8.0, 9.0, 9.6):
    i = int(np.argmin(np.abs(xc - xs)))
    a2.plot(t[ult] - t[ult][0], u_eje[ult][:, i], label=f"x = {xs} cm")
a2.axhline(0, color=GRIS_MEDIO, lw=0.8)
a2.set(xlabel="t dentro de la onda [s]", ylabel="u axial [cm/s]", title="Cerca del píloro: avance y retropulsión")
a2.legend(fontsize=9)
fig.tight_layout(); plt.show()

u_retro = float(-np.nanmin(u_eje[ult][:, xc > 7.0]))
print(f"Velocidad máxima del chorro retropulsivo: {u_retro:.2f} cm/s  = {u_retro / pa.c:.1f} veces la velocidad de la onda")
print(f"Reynolds del chorro (u · H/2 / ν): {u_retro * pa.H / 2 / pa.nu:.1f}")
print(f"Cizalla máxima en el último periodo: {np.nanmax(SIM['tau_max'][ult]):.3f} Pa")

In [ ]:
# Animación de una onda (rapidez + flechas). ANIMAR = False para ahorrar tiempo.
ANIMAR = True
if ANIMAR:
    fig, a = plt.subplots(figsize=(8, 3.6))
    vmax = np.nanpercentile(rapidez[ult], 99.5)
    k0 = ult.start
    im = a.imshow(rapidez[k0], origin="lower", extent=[xc[0], xc[-1], yc[0], yc[-1]], cmap="Greys", vmin=0, vmax=vmax, aspect="equal")
    q = a.quiver(XC[::5, ::5], YC[::5, ::5], np.nan_to_num(SIM["ux"][k0][::5, ::5]), np.nan_to_num(SIM["uy"][k0][::5, ::5]), color=C1, scale=6, width=0.004)
    plt.colorbar(im, ax=a, shrink=0.8, label="rapidez [cm/s]"); a.grid(False)
    tit = a.set_title("")
    def frame(n):
        k = k0 + n
        im.set_data(rapidez[k]); q.set_UVC(np.nan_to_num(SIM["ux"][k][::5, ::5]), np.nan_to_num(SIM["uy"][k][::5, ::5]))
        tit.set_text(f"Antro — t = {t[k] - t[k0]:.1f} s de la onda")
        return im, q, tit
    anim = animation.FuncAnimation(fig, frame, frames=range(0, k_per + 1, 2), interval=100)
    plt.close(fig)
    display(HTML(anim.to_jshtml()))

## 5. ¿Cuánto mezcla el flujo real? Trazadores y la $D_{mix}$ de la fase 2

Soltamos 3000 partículas pasivas, que siguen el flujo con interpolación bilineal y Heun en el tiempo. Las observamos **estroboscópicamente**, una vez por onda, para eliminar el vaivén periódico y quedarnos con la dispersión neta. Si la dispersión fuera difusiva, cumpliría
$$ \langle (\Delta x)^2 \rangle \approx 2\,D_{ef}\,t . $$
Este cociente nos da una estimación de la difusividad efectiva que producen las ondas, comparable con el $D_{mix}$ que la fase 2 tuvo que suponer.

In [ ]:
px, py = SIM["px"] * dx, SIM["py"] * dx
i_ref = int(np.argmin(np.abs(t - pa.T)))         # posición tras la onda de arranque
filas = []
for n in range(1, PERIODOS):
    i1 = int(np.argmin(np.abs(t - (n + 1) * pa.T)))
    d2x = float(np.mean((px[i1] - px[i_ref]) ** 2)); d2y = float(np.mean((py[i1] - py[i_ref]) ** 2))
    filas.append({"ondas": n, "<Δx²> [cm²]": d2x, "D_ef axial [cm²/min]": d2x / (2 * n * pa.T) * 60,
                  "D_ef transversal [cm²/min]": d2y / (2 * n * pa.T) * 60})
DEF = pd.DataFrame(filas)
display(DEF.style.format(precision=3).hide(axis="index"))

fig, axs = plt.subplots(1, 2, figsize=(12, 3.6))
color_ini = np.where(px[0] < 5, C1, C2)
for a, k, tit in [(axs[0], 0, "Posición inicial"), (axs[1], len(t) - 1, f"Tras {PERIODOS} ondas")]:
    a.scatter(px[k], py[k], s=2, c=color_ini, alpha=0.7, lw=0)
    a.contour(XC, YC, SIM["solido"][k].astype(float), levels=[0.5], colors=GRIS, linewidths=0.8)
    a.set(xlim=(0, pa.Lx), ylim=(0, pa.H + 2 * pa.margen), aspect="equal", title=tit, xlabel="x [cm]"); a.grid(False)
axs[0].set_ylabel("y [cm]")
fig.suptitle("Trazadores (azul: mitad proximal; naranja: mitad distal)", fontweight="bold"); fig.tight_layout(); plt.show()

**Lectura.** Los trazadores forman **estriaciones**: el flujo estira y lamina el contenido, sobre todo en el antro, pero en tres ondas todavía no lo mezcla a escala fina. La dispersión crece más deprisa que linealmente durante las primeras ondas: aún no estamos en régimen difusivo, porque el flujo, casi periódico y de Reynolds bajo, estira los filamentos antes de mezclarlos. Por eso la $D_{ef}$ obtenida es una **estimación de orden de magnitud**: unas décimas de cm²/min en la dirección axial, y menos en la transversal. Cae dentro del rango que exploró la fase 2 (0.05–2 cm²/min) y algo por debajo del valor por defecto de allí, 0.5 cm²/min. Para afinarla harían falta más ondas y una malla más fina, que es justo para lo que sirve la GPU.

## 6. GPU: JAX o CuPy

El núcleo no modifica arrays en el sitio, así que el mismo código corre en GPU:
- **JAX**: `jax.jit` compila el bloque de pasos completo con `lax.fori_loop`, y la GPU recorre el bucle de tiempo sin volver a Python.
- **CuPy**: misma API que NumPy, solo cambia `xp`. Cada paso lanza sus kernels desde Python.

La celda siguiente:
1. Detecta qué hay disponible.
2. Ejecuta un **test de equivalencia**: 300 pasos del antro en GPU (precisión simple) contra NumPy (doble). Deben coincidir salvo redondeo de precisión simple.
3. Mide la velocidad.

Sin GPU, la celda informa y no hace nada.

In [ ]:
def probar_backend(nombre):
    if nombre == "jax":
        import jax, jax.numpy as jnp
        if not any(d.platform == "gpu" for d in jax.devices()):
            print("JAX disponible, pero sin GPU: se usará CPU (sirve para el test, no para la velocidad)")
        Rg = Red(jnp, "float32")
    else:
        import cupy as cp
        Rg = Red(cp, "float32")
    geom_g, nx_g, ny_g, Ug = geometria_antro(pa, Rg)
    paso_g = crear_paso(Rg, pa.tau, geom_g)
    N = 300
    f0 = Rg.w * Rg.xp.ones((9, ny_g, nx_g), dtype="float32")
    if nombre == "jax":
        import jax
        bloque = jax.jit(lambda f, t0: integrar(paso_g, f, t0, N, Rg, usar_jax=True))
        fg = bloque(f0, 0.0); fg.block_until_ready()            # primera llamada: compila
        t0 = time.time(); fg2 = bloque(fg, float(N)); fg2.block_until_ready(); dt_gpu = (time.time() - t0) / N
    else:
        fg = integrar(paso_g, f0, 0, N, Rg)
        import cupy as cp; cp.cuda.Stream.null.synchronize()
        t0 = time.time(); fg2 = integrar(paso_g, fg, N, N, Rg); cp.cuda.Stream.null.synchronize(); dt_gpu = (time.time() - t0) / N
    # referencia NumPy float64
    paso_c = crear_paso(R, pa.tau, geometria_antro(pa, R)[0])
    t0 = time.time(); fc = integrar(paso_c, R.w * np.ones((9, ny_g, nx_g)), 0, N, R); dt_cpu = (time.time() - t0) / N
    _, uxg, uyg = macroscopicas(fg, Rg); _, uxc, uyc = macroscopicas(fc, R)
    s = a_numpy(geom_g(N)[0])
    dif = np.nanmax(np.abs(np.where(s, 0, a_numpy(uxg) - uxc))) / np.max(np.abs(np.where(s, 0, uxc)))
    return dif, dt_cpu, dt_gpu

for nombre in ("jax", "cupy"):
    try:
        dif, dcpu, dgpu = probar_backend(nombre)
        print(f"{nombre}: diferencia relativa con NumPy = {dif:.1e}   |   CPU {dcpu*1e3:.2f} ms/paso, {nombre} {dgpu*1e3:.3f} ms/paso  (×{dcpu/dgpu:.0f})")
        VERIFICACION.append({"test": f"T7 equivalencia {nombre} (float32) vs NumPy", "valor": dif, "criterio": "< 1e-3", "pasa": dif < 1e-3})
    except ImportError:
        print(f"{nombre}: no está instalado en este entorno (en Colab con GPU, JAX y CuPy vienen preinstalados)")

**Simulación de alta resolución en GPU.** Si la GPU está disponible, puedes repetir la sección 4 con más resolución, o con una viscosidad más cercana al agua, cambiando los parámetros. Un ejemplo:
```python
pa_hd = ParamAntro(nx=256, nu=0.05, tau=0.6)          # malla 2× y comida más fluida
import jax, jax.numpy as jnp
Rg = Red(jnp, "float32")
SIM_HD = simular_antro(pa_hd, Rg, periodos=4, usar_jax=True)
```
`simular_antro` devuelve arrays de NumPy, así que todas las celdas de análisis anteriores funcionan sin cambios. Antes de bajar la viscosidad, comprueba que el Mach máximo (test T6) siga por debajo de 0.1. Si no, aumenta `nx` o `tau`.

## 7. Exportar el campo de velocidades para el transporte

La siguiente fase acoplará este flujo al transporte químico de la fase 2. Guardamos el último periodo: velocidades, máscara sólida, tiempos y parámetros, en un `.npz` comprimido.

In [ ]:
np.savez_compressed("campo_antro.npz", t=t[ult] - t[ult][0], ux=np.nan_to_num(SIM["ux"][ult]), uy=np.nan_to_num(SIM["uy"][ult]),
                    solido=SIM["solido"][ult], x=xc, y=yc, dx=dx, periodo=pa.T, nu=pa.nu,
                    D_ef_axial_cm2_min=float(DEF["D_ef axial [cm²/min]"].iloc[-1]))
import os
print(f"campo_antro.npz: {os.path.getsize('campo_antro.npz') / 1e6:.1f} MB")

## 8. Resumen de verificación

In [ ]:
ver = pd.DataFrame(VERIFICACION)
ver["resultado"] = np.where(ver["pasa"], "✅ pasa", "❌ FALLA")
display(ver.drop(columns="pasa").style.format({"valor": "{:.3g}"}).hide(axis="index"))
n_ok = int(ver["pasa"].sum())
print(f"\n{n_ok}/{len(ver)} tests superados")
assert n_ok == len(ver), "Hay tests de verificación que no pasan"

## 9. Siguientes pasos

**Establecido en esta fase**
- Tenemos un solucionador Lattice Boltzmann D2Q9 verificado: orden 2 con paredes fijas, pared móvil exacta en Couette, viscosidad correcta y masa conservada.
- Con paredes que se deforman, reproduce la teoría de bombeo peristáltico de Shapiro con un error de alrededor del 1 %.
- En el antro aparecen el chorro retropulsivo, los vórtices detrás de cada contracción y las zonas de alta cizalla junto al estrechamiento. Este último es el mecanismo físico de la trituración que planteaste al principio.
- La mezcla real estimada con trazadores es de unas décimas de cm²/min, del orden del $D_{mix}$ que la fase 2 tuvo que suponer.

**Siguiente fase (4): acoplamiento.**
1. Usar `campo_antro.npz` como velocidad del transporte de la fase 2. Hará falta proyectar el campo a una malla de volúmenes finitos libre de divergencia y tratar las celdas que se abren y cierran.
2. Abrir el píloro, que deje pasar líquido y partículas pequeñas, para recuperar el **vaciado** y volver a evaluar el registro de validación de la fase 1b.
3. Partículas de comida que se rompen cuando la cizalla supera su resistencia, que además se debilita con el ácido: el acoplamiento químico-mecánico completo.